# Section 20 — The Malware Dataset

HTB's **malimg** data contains PNG images grouped into folders named for 25 malware families. We inspect images, not executable malware files. An original grayscale pixel value represents a byte value from 0 (black) through 255 (white). Read `p_i = b_i` as **pixel sub i equals byte sub i**: brightness at position `i` stores the corresponding byte. Bytes `[0, 128, 255]` map to black, mid-gray, and white. Exact recovery would require preserving the original pixels, their order, and the original byte length; resizing or lossy conversion would break this property.

This notebook implements only this section's **dataset inspection and family-frequency plot**. It does not train a CNN or execute malware. The HTB-linked ZIP stays in Downloads and is never added to Git. We read the ZIP directly, avoiding a 1.09 GB extraction.

In [ ]:
import os
from pathlib import Path, PurePosixPath
from collections import Counter
import zipfile

# Override MALIMG_ZIP if you stored the HTB-linked archive elsewhere.
archive_path = Path(os.environ.get('MALIMG_ZIP', Path.home() / 'Downloads' / 'malimg-original.zip'))
if not archive_path.is_file():
    raise FileNotFoundError(f'Download the HTB-linked malimg ZIP, then set MALIMG_ZIP: {archive_path}')
if not zipfile.is_zipfile(archive_path):
    raise ValueError(f'Not a valid ZIP archive: {archive_path}')
print('Archive:', archive_path.name, '| bytes:', archive_path.stat().st_size)

## Count family labels without extracting the archive

Each ZIP path has the form `malimg_paper_dataset_imgs/<family>/<sample>.png`. The middle component is the label. A **class frequency** counts images with that label. In notation, `f_c = Σ_i 1[y_i = c]`; say **f sub c equals the sum over images i of an indicator that y sub i equals c**. `y_i` is image `i`'s folder label, `c` is one family, and the indicator contributes 1 on a match or 0 otherwise. In ordinary English, count filenames in family `c`. For labels `[A, B, A]`, family A has `1 + 0 + 1 = 2` images. We validate member paths before counting; listing ZIP names does not execute contents.

In [ ]:
with zipfile.ZipFile(archive_path) as archive:
    members = [member for member in archive.infolist() if not member.is_dir()]

# Each PNG should sit directly below exactly one family folder.
paths = [PurePosixPath(member.filename) for member in members]
if not all(len(path.parts) == 3 and path.parts[0] == 'malimg_paper_dataset_imgs'
           and path.suffix.lower() == '.png' and '..' not in path.parts for path in paths):
    raise ValueError('Unexpected ZIP member; inspect the archive before using it.')
family_counts = Counter(path.parts[1] for path in paths)
image_count = sum(family_counts.values())
assert image_count == len(paths)
print('Images:', image_count, '| families:', len(family_counts))
print('Five largest families:', family_counts.most_common(5))
print('Five smallest families:', sorted(family_counts.items(), key=lambda item: item[1])[:5])
if image_count != 9339 or len(family_counts) != 25:
    print('Note: this archive differs from the 9,339-image/25-family version HTB describes.')

## Why the counts matter

The **class share** for family `c` is `f_c / N`; say **frequency of class c divided by capital N**, where `N` is the total image count. Here `Allaple.A` has 2,949 of 9,339 images, about 31.6%, while `Skintrim.N` has 80, about 0.86%. If a classifier always guessed the largest family, it would be right about 31.6% of the time without learning the other 24 families. The plot makes this imbalance visible. Balancing and training belong to later sections; we do not change the data here.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

ordered = sorted(family_counts.items(), key=lambda item: item[1])
families = [family for family, count in ordered]
frequencies = [count for family, count in ordered]
fig, ax = plt.subplots(figsize=(10, 9), facecolor='#141D2B')
ax.set_facecolor('#141D2B')
ax.barh(families, frequencies, color='#9FEF00', edgecolor='black')
ax.set(title='Malware Family Class Distribution', xlabel='Number of images', ylabel='Malware family')
ax.tick_params(colors='#A4B1CD')
ax.xaxis.label.set_color('#9FEF00')
ax.yaxis.label.set_color('#9FEF00')
ax.title.set_color('#9FEF00')
fig.tight_layout()
display(fig)  # Jupyter displays the chart without a blocking GUI window.
plt.close(fig)  # Release chart memory after display.

## Inspect one image's metadata, not a malware program

`L` is Pillow's single-channel 8-bit grayscale mode. We check one bounded-size PNG for its mode and dimensions without extracting or executing files. Image dimensions may vary. HTB has not yet specified a model input shape, so we do not resize or normalize anything here.

In [ ]:
from io import BytesIO
from PIL import Image

sample_member = members[0]
if sample_member.file_size > 5_000_000:
    raise ValueError('Sample PNG exceeds the small inspection limit.')
with zipfile.ZipFile(archive_path) as archive:
    sample_bytes = archive.read(sample_member)
with Image.open(BytesIO(sample_bytes)) as sample_image:
    print('Sample family:', PurePosixPath(sample_member.filename).parts[1])
    print('Image mode:', sample_image.mode, '| width × height:', sample_image.size)
    print('Top-left pixel brightness:', sample_image.getpixel((0, 0)))
    assert sample_image.mode == 'L'
print('Section 20 complete: archive structure and class distribution inspected; no model trained.')

# Section 21 — Preprocessing the Malware Dataset

HTB makes an **80% training / 20% test** split, resizes images to 75×75, converts them to tensors, normalizes RGB channels, and wraps them in DataLoaders. We use the same 80/20 per-family split and transform, but save memory/disk by storing train/test **indices** rather than copying all PNGs into another directory. The course uses `split-folders` and `ImageFolder` on copied `train`/`test` folders; here one safe extraction to Downloads lets `ImageFolder` read the originals and `Subset` selects disjoint groups. The extracted directory is not in Git.

Normalization for one channel is `z = (v − μ) / σ`. Say **z equals value v minus mean mu, divided by standard deviation sigma**. `v` is the pixel after `ToTensor` scales byte 0–255 into 0–1; `μ` (mu) and `σ` (sigma) are fixed per-channel constants. For red-channel `v=0.5`, `(0.5−0.485)/0.229≈0.066`. Complete meaning: shift and scale the channel values into the numerical range this model input convention expects. `ImageFolder`'s default loader converts the grayscale PNG to RGB, which is necessary for HTB's three-entry normalization lists; the three RGB channels begin as copies of the grayscale value. Resizing loses fine byte-level detail, so the 75×75 tensor is **not** a reversible binary representation.

HTB uses batches of 1,024 and two workers. On this Windows laptop we use 64 and zero workers to avoid large prefetched tensors and extra Python processes. This changes throughput, not labels or image transform. No CNN is trained yet.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder

def load_datasets(base_path, train_batch_size=64, test_batch_size=64, seed=1337):
    # ImageFolder discovers family subfolders and maps their names to integer labels.
    # Its default PIL loader converts grayscale PNGs to RGB (three channels).
    transform = transforms.Compose([
        transforms.Resize((75, 75)),
        transforms.ToTensor(),  # byte 0..255 becomes float 0..1; shape C×H×W
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),  # z=(v−mu)/sigma per channel
    ])
    all_images = ImageFolder(root=str(base_path), transform=transform)
    rng = np.random.default_rng(seed)
    train_indices, test_indices = [], []
    # Split within each family, so both partitions contain each class.
    for class_id in range(len(all_images.classes)):
        class_indices = np.flatnonzero(np.asarray(all_images.targets) == class_id)
        shuffled = rng.permutation(class_indices)
        cut = int(0.8 * len(shuffled))  # floor, matching HTB's 7,459/1,880 counts
        train_indices.extend(shuffled[:cut].tolist())
        test_indices.extend(shuffled[cut:].tolist())
    if set(train_indices) & set(test_indices):
        raise AssertionError('Train/test index overlap')
    if len(train_indices) + len(test_indices) != len(all_images):
        raise AssertionError('A sample was lost during splitting')
    train_dataset = Subset(all_images, train_indices)
    test_dataset = Subset(all_images, test_indices)
    # Zero workers avoids Windows subprocess duplication and prefetch memory.
    loader_rng = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size,
                              shuffle=True, num_workers=0, generator=loader_rng)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size,
                             shuffle=False, num_workers=0)
    return train_loader, test_loader, len(all_images.classes)

In [ ]:
# The Section 20 ZIP was safely extracted once to this Downloads directory.
# Override MALIMG_IMAGE_DIR if your extracted family folders live elsewhere.
image_root = Path(os.environ.get('MALIMG_IMAGE_DIR',
                                 Path.home() / 'Downloads' / 'malimg_paper_dataset_imgs'))
if not image_root.is_dir():
    raise FileNotFoundError(f'Extract the HTB-linked ZIP once before this section: {image_root}')
train_loader, test_loader, n_classes = load_datasets(image_root)
train_dataset, test_dataset = train_loader.dataset, test_loader.dataset
print('Train:', len(train_dataset), '| test:', len(test_dataset), '| classes:', n_classes)
assert (len(train_dataset), len(test_dataset), n_classes) == (7459, 1880, 25)
assert train_dataset.dataset.class_to_idx == test_dataset.dataset.class_to_idx
images, labels = next(iter(train_loader))  # Load one small batch, not all 9,339 images.
print('One batch shape:', tuple(images.shape), '| label shape:', tuple(labels.shape))
assert images.shape == (64, 3, 75, 75)  # batch, RGB channels, height, width
assert labels.shape == (64,) and torch.isfinite(images).all()
assert labels.min() >= 0 and labels.max() < n_classes

## What did resizing change?

The next cell shows one original PNG and its resized model input. We *undo normalization only for display* using `v = z×σ + μ`, read **v equals z times sigma plus mu**. This is algebraically the reverse of `z=(v−μ)/σ`: multiply by sigma and add mu. It does not undo resizing or recover the original binary bytes. HTB plots the normalized tensor directly, which can look oddly colored or clipped because those values are no longer ordinary 0–1 RGB intensities. The model still receives the normalized tensor, not the display copy.

In [ ]:
from PIL import Image

first_index = train_dataset.indices[0]
first_path, first_class_id = train_dataset.dataset.samples[first_index]
normalized_image, label_id = train_dataset[0]
assert label_id == first_class_id
mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
display_image = (normalized_image * std + mean).clamp(0, 1).permute(1, 2, 0)
with Image.open(first_path) as original:
    original_gray = original.copy()
fig, axes = plt.subplots(1, 2, figsize=(9, 4), facecolor='#141D2B')
axes[0].imshow(original_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'Original: {train_dataset.dataset.classes[label_id]}')
axes[1].imshow(display_image)
axes[1].set_title('75×75 RGB model input (display-normalized)')
for ax in axes:
    ax.axis('off')
fig.tight_layout()
display(fig)
plt.close(fig)
print('Section 21 complete: disjoint loaders prepared; no model trained.')

# Section 22 — The Model

HTB starts with a **pretrained ResNet50** image classifier. Its many convolutional layers act as a feature extractor; their ImageNet-trained weights are reused, then a new final head learns our malware-family labels. **Freeze** means setting each backbone parameter's `requires_grad=False`: gradient-based training will not update those weights. We replace the original final layer with `Linear(2048,1000) → ReLU → Linear(1000,n_classes)`. A linear layer forms weighted sums; ReLU means `max(0,a)`, said **the larger of zero and a**. For inputs `−2` and `3`, ReLU gives `0` and `3`.

For a batch of `B` images, the output has shape `B×25`, said **batch by 25 class scores**. Each score is a **logit**—a raw score before probabilities. The biggest logit indicates the current predicted family; we do not add softmax here because the later training loss may expect logits. The new head starts with random weights even though the backbone is pretrained, so these initial predictions have no validated malware meaning. HTB calls this fine-tuning; more precisely it trains a new head while keeping backbone weights fixed.

Torchvision's current `ResNet50_Weights.DEFAULT` resolves to its ImageNet-1K V2 weights, which are already cached on this machine. HTB's 75×75 preprocessing is smaller than the pretrained weights' standard 224×224 inference recipe; it is the course's proof-of-concept choice, not a claim that pretrained ImageNet accuracy transfers unchanged. We instantiate and do only a two-image shape check here—**no loss, optimizer, or training loop yet**.

In [ ]:
import torch.nn as nn
from torchvision import models

HIDDEN_LAYER_SIZE = 1000

class MalwareClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        # HTB uses pretrained ResNet50; torchvision may fetch these weights once.
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Freeze the original feature extractor before replacing its classifier.
        for parameter in self.resnet.parameters():
            parameter.requires_grad = False
        num_features = self.resnet.fc.in_features  # 2048 features per image
        # Only these newly created head layers retain requires_grad=True.
        self.resnet.fc = nn.Sequential(
            nn.Linear(num_features, HIDDEN_LAYER_SIZE),
            nn.ReLU(),  # ReLU(a) = max(0, a)
            nn.Linear(HIDDEN_LAYER_SIZE, n_classes),
        )

    def forward(self, x):
        # x: B×3×75×75; return: B×n_classes raw logits.
        return self.resnet(x)

In [ ]:
torch.manual_seed(1337)  # Reproducible initialization of the new head.
model = MalwareClassifier(n_classes)  # n_classes comes from ImageFolder, not a guess.
frozen = [p for name, p in model.named_parameters() if not name.startswith('resnet.fc.')]
head = list(model.resnet.fc.parameters())
assert frozen and head and all(not p.requires_grad for p in frozen)
assert all(p.requires_grad for p in head)
assert model.resnet.fc[0].in_features == 2048
assert model.resnet.fc[-1].out_features == n_classes
model.eval()  # Avoid changing batch-normalization running statistics during this check.
with torch.inference_mode():
    sample_logits = model(images[:2])  # Two already-preprocessed training images only.
assert sample_logits.shape == (2, n_classes)
assert torch.isfinite(sample_logits).all()
print('Logit shape:', tuple(sample_logits.shape))
print('Frozen backbone parameters:', sum(p.numel() for p in frozen))
print('Trainable head parameters:', sum(p.numel() for p in head))
print('Section 22 complete: pretrained backbone and new head verified; no training yet.')

# Section 23 — Training and Evaluation (Malware Image Classification)

For each image, **cross-entropy loss** rewards a high logit for the true family relative to the other 24. For one image it can be written `L = −log(exp(z_y) / Σ_j exp(z_j))`. Read this aloud: **L equals negative log of e to the true-class logit z sub y, divided by the sum over all classes j of e to each logit z sub j**. `z_y` is the true-family raw score, the denominator collects all class scores after exponentiation, and the fraction is the softmax probability assigned to the true family. Complete meaning: penalize the model when it assigns little probability to the correct family. For two-class logits `[2,0]` with class 0 correct, that probability is about 0.881 and the loss about 0.127. PyTorch's `CrossEntropyLoss` computes this stably **from logits**, so do not apply softmax first.

A gradient is the direction in which loss changes as a weight changes. `loss.backward()` computes those directions for the new head, and Adam's `step()` updates it. The backbone has no trainable weight gradients. We additionally keep its batch-normalization running statistics fixed in evaluation mode while the head trains; HTB freezes parameters but does not address those buffers. This is a labeled implementation improvement.

An **epoch** is one pass through all 7,459 training images. Accuracy is `100×n_correct/n_total`, read **one hundred times the number correct divided by the total number seen**. If 8 of 10 are correct, accuracy is 80%. Training accuracy is not an independent test score. HTB demonstrates ten epochs and reports 88.54% test accuracy on its own split; our split and resource settings may differ. The full run is opt-in to avoid surprise compute on the laptop. The test set is evaluated only after training; do not tune repeatedly on its result.

In [ ]:
import time

def compute_accuracy(n_correct, n_total):
    if n_total == 0:
        raise ValueError('Cannot compute accuracy with no examples.')
    return 100.0 * n_correct / n_total

def train(model, train_loader, n_epochs, device, verbose=False):
    model.to(device)
    criterion = nn.CrossEntropyLoss()  # Stable log-softmax plus true-class penalty.
    optimizer = torch.optim.Adam(model.resnet.fc.parameters())  # Only trainable head.
    history = {'accuracy': [], 'loss': []}
    for epoch in range(n_epochs):
        model.train()
        model.resnet.eval()  # Preserve frozen backbone batch-norm statistics.
        model.resnet.fc.train()  # Keep the new head in training mode.
        loss_sum, n_correct, n_total = 0.0, 0, 0
        started = time.perf_counter()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)  # B×25 raw scores, one row per image.
            loss = criterion(logits, labels)  # Mean loss for this batch.
            loss.backward()  # Gradients for the unfrozen head parameters.
            optimizer.step()  # Adam changes only those head parameters.
            batch_size = labels.size(0)
            n_total += batch_size
            n_correct += (logits.argmax(dim=1) == labels).sum().item()
            loss_sum += loss.item() * batch_size  # Weight the shorter last batch correctly.
        history['accuracy'].append(compute_accuracy(n_correct, n_total))
        history['loss'].append(loss_sum / n_total)
        if verbose:
            print(f'Epoch {epoch + 1}/{n_epochs}: train acc={history["accuracy"][-1]:.2f}% '
                  f'loss={history["loss"][-1]:.4f}, seconds={time.perf_counter() - started:.1f}',
                  flush=True)
    return history

In [ ]:

def predict(model, inputs):
    model.eval()
    with torch.inference_mode():
        return model(inputs).argmax(dim=1)

def evaluate(model, test_loader, device):
    model.to(device).eval()
    n_correct, n_total = 0, 0
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predicted = predict(model, inputs)
        n_correct += (predicted == labels).sum().item()
        n_total += labels.size(0)
    return compute_accuracy(n_correct, n_total)

def save_model(model, path):
    # HTB uses torch.jit.script in a .py file. Interactive notebook classes may lack
    # inspectable source, so tracing this fixed-shape, branch-free forward is reliable.
    model = model.cpu().eval()
    example = torch.zeros(1, 3, 75, 75)
    with torch.inference_mode():
        traced = torch.jit.trace(model, example, check_trace=False)
        assert torch.allclose(traced(example), model(example), atol=1e-5)
    traced.save(str(path))
    return path

## Check gradients before a long run

The tiny check below computes one cross-entropy and backward pass on two images. It does **not** call `optimizer.step()`, so the model weights remain unchanged. It verifies the head receives finite gradients while frozen backbone parameters do not. This catches shape/label/gradient errors before a ten-epoch training job.

In [ ]:
model.train()
model.resnet.eval()
model.resnet.fc.train()
probe_logits = model(images[:2])
probe_loss = nn.CrossEntropyLoss()(probe_logits, labels[:2])
probe_loss.backward()
assert torch.isfinite(probe_loss)
assert all(p.grad is None for p in frozen)
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in head)
model.zero_grad(set_to_none=True)
model.eval()
print('Two-image cross-entropy and head-only gradients passed:', float(probe_loss))

## Opt-in full training, single test evaluation, and model export

HTB runs 10 epochs, saves TorchScript, evaluates once on test, and plots training curves. The code below implements that workflow with the smaller Section 21 batches and zero workers. `RUN_FULL_TRAINING=False` prevents a costly Run All from silently using the GPU; set it to `True` when ready. The model is saved to your Downloads folder, **outside Git**. TorchScript `script` fails for some notebook-defined classes because their source is not inspectable; this notebook uses `trace` for the simple branch-free `forward` and checks its output. A traced model is tied to the example's 75×75 preprocessing contract; use the same transform at inference. A saved model can be loaded by `torch.jit.load`, but only load your own trusted artifact.

The plots show **training** curves only. A high training score does not prove test performance or detection of unseen malware families. HTB's 88.54% is its example result, not a promised score for this seeded split.

In [ ]:
RUN_FULL_TRAINING = False  # Explicitly enable the GPU/CPU training job when ready.
N_EPOCHS = 10
# Override for a writable local path; default stays outside the Git checkout.
MODEL_FILE = Path(os.environ.get('MALWARE_MODEL_FILE',
                                 Path.home() / 'Downloads' / 'malware_classifier.pth'))
TRAIN_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if RUN_FULL_TRAINING:
    print('Training device:', TRAIN_DEVICE, '| epochs:', N_EPOCHS, flush=True)
    training_information = train(model, train_loader, N_EPOCHS, TRAIN_DEVICE, verbose=True)
    test_accuracy = evaluate(model, test_loader, TRAIN_DEVICE)  # One reserved-test pass.
    print(f'Test accuracy: {test_accuracy:.2f}%', flush=True)
    saved_path = save_model(model, MODEL_FILE)
    print('Saved trusted TorchScript model:', saved_path, flush=True)
    for key, title, units in [('accuracy', 'Training Accuracy', 'Accuracy (%)'),
                              ('loss', 'Training Loss', 'Cross-entropy loss')]:
        fig, ax = plt.subplots(figsize=(8, 4), facecolor='#141D2B')
        ax.set_facecolor('#141D2B')
        ax.plot(range(1, N_EPOCHS + 1), training_information[key], color='#9FEF00')
        ax.set(title=title, xlabel='Epoch', ylabel=units)
        ax.tick_params(colors='#A4B1CD')
        fig.tight_layout()
        display(fig)
        plt.close(fig)
else:
    print('Full training disabled; gradient smoke test passed and no model was saved.')